# 01. Core Loop & ReAct

**arcrun is the runtime that makes an LLM into an agent.** Given a model (from arcllm), a set of tools, a system prompt, and a task, `run()` keeps the model talking, calling tools, and observing results until the task is finished — or a budget cap fires.

ReAct is the default execution strategy: **Reason → Act → Observe → Repeat.** The model proposes tool calls, the loop dispatches them, the results come back as messages, and the model takes the next step.

This notebook covers everything that happens inside one `run()` call:

- the ReAct loop in 60 seconds — what the strategy actually does turn-by-turn
- defining a `Tool` and what it sees via `ToolContext`
- the `task_complete` builtin — structured early exit (NEW in v0.4.0)
- `max_turns` and `max_cost_usd` — hard budget caps that synthesize a `failed` payload (NEW in v0.4.0)
- the `loop.completed` event — what fires and what's in the payload (NEW in v0.4.0)
- hash-chained event integrity via `verify_chain()` — tamper-evident audit
- strategy prompt injection via `get_strategy_prompts()` — what the model is told (NEW in v0.4.0)

Mock-first: every cell runs without an API key. The single `(live)` section is gated on `ANTHROPIC_API_KEY` and skips cleanly when missing.

## 1. Setup

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

We also flag whether a real API key is present. The mock cells run regardless; only the `(live)` section uses it.

In [ ]:
# (live) optional — set this to run real-API sections; mock cells run regardless
HAS_LIVE_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print(f"Live API key: {'present' if HAS_LIVE_KEY else 'missing — live cells will skip'}")

Pull the public surface of arcrun into scope. Notice the `RunHandle` from `loop`, the `task_complete` builtin from `builtins`, and the event-chain primitives from `events`.

In [ ]:
import arcrun
from arcrun import (
    Event,
    EventBus,
    LoopResult,
    RunHandle,
    SandboxConfig,
    Strategy,
    Tool,
    ToolContext,
    ToolRegistry,
    get_strategy_prompts,
    run,
    run_async,
    verify_chain,
)
from arcrun.builtins.task_complete import (
    TaskCompleteArgs,
    make_budget_breach_args,
    make_task_complete_tool,
)

print(f"arcrun {arcrun.__version__}")
print(
    "Public symbols loaded:",
    [
        "run",
        "run_async",
        "RunHandle",
        "Strategy",
        "LoopResult",
        "Tool",
        "ToolContext",
        "ToolRegistry",
        "SandboxConfig",
        "Event",
        "EventBus",
        "verify_chain",
        "get_strategy_prompts",
        "make_task_complete_tool",
        "TaskCompleteArgs",
    ],
)

Define a tiny `MockModel` (mirrors the one in `arcrun/tests/conftest.py`) so the notebook is fully deterministic — no network calls, no flakes. It plays back a scripted list of `LLMResponse` objects, one per `model.invoke()` call.

In [ ]:
from dataclasses import dataclass, field
from typing import Any


@dataclass
class Usage:
    input_tokens: int = 10
    output_tokens: int = 5
    total_tokens: int = 15


@dataclass
class ToolCall:
    id: str
    name: str
    arguments: dict[str, Any]


@dataclass
class LLMResponse:
    content: str | None = None
    tool_calls: list[ToolCall] = field(default_factory=list)
    stop_reason: str = "end_turn"
    usage: Usage = field(default_factory=Usage)
    cost_usd: float = 0.001


class MockModel:
    """Replays a scripted list of LLMResponse objects, one per invoke."""

    def __init__(self, responses: list[LLMResponse]) -> None:
        self._responses = list(responses)
        self._idx = 0

    async def invoke(self, messages: list, tools: list | None = None) -> LLMResponse:
        if self._idx >= len(self._responses):
            # If under-specified, keep ending the turn so demos don't hang.
            return LLMResponse(content="(mock exhausted)", stop_reason="end_turn")
        resp = self._responses[self._idx]
        self._idx += 1
        return resp


print("MockModel ready.")

## 2. ReAct in 60 seconds

The ReAct strategy is the default execution loop. Each iteration is one *turn*:

```
while turn_count < max_turns and cost_usd < max_cost_usd:
    1. (cost cap check — synthesize failed completion if breached)
    2. emit turn.start
    3. drain any steer messages into the conversation
    4. transform_context(messages) if hook provided
    5. response = await model.invoke(messages, tools)
    6. accumulate token usage and cost
    7. emit llm.call
    8. append assistant message (text + tool_use blocks)
    9. if stop_reason == 'end_turn' and no tool_calls:
        - drain followup queue (continue) OR return LoopResult
    10. dispatch tool calls (parallel_safe → asyncio.gather, otherwise serial)
    11. if any tool was signals_completion=True: emit loop.completed, return
    12. emit turn.end
```

Three things terminate the loop cleanly:

1. **`stop_reason == 'end_turn'` with no tool calls** — the model is finished talking.
2. **A tool flagged `signals_completion=True` fires** — usually `task_complete`.
3. **A budget cap is breached** — `max_turns` or `max_cost_usd` synthesizes a `failed` completion payload.

The strategy itself is a dataclass with `name`, `description`, `prompt_guidance`, and an async `__call__`. Inspect the built-in ReAct strategy.

In [ ]:
from arcrun.strategies import STRATEGIES, _load_strategies

if not STRATEGIES:
    _load_strategies()

react = STRATEGIES["react"]
print(f"name:        {react.name}")
print(f"description: {react.description}")
print()
print("prompt_guidance (this is what the model sees):")
print("-" * 60)
print(react.prompt_guidance)

## 3. First loop run

Define one trivial tool and drive the loop with a `MockModel`. The mock returns a single `end_turn` response with no tool calls — the simplest possible run.

In [ ]:
async def _greet(params: dict, ctx: ToolContext) -> str:
    return f"hello, {params.get('name', 'world')}"


greet_tool = Tool(
    name="greet",
    description="Return a greeting for the given name.",
    input_schema={
        "type": "object",
        "properties": {"name": {"type": "string"}},
        "required": ["name"],
    },
    execute=_greet,
)

model = MockModel(
    [
        LLMResponse(
            content="Nothing to do — task is already trivially done.", stop_reason="end_turn"
        ),
    ]
)

result = await run(
    model=model,
    tools=[greet_tool],
    system_prompt="You are a concise assistant.",
    task="Say hello.",
)

print("content:        ", result.content)
print("turns:          ", result.turns)
print("tool_calls_made:", result.tool_calls_made)
print("strategy_used:  ", result.strategy_used)
print("cost_usd:       ", result.cost_usd)
print("events emitted: ", len(result.events))

`run()` returns a `LoopResult` — a frozen record of the entire execution. Walk the event timeline so you can see the order things happened in.

In [ ]:
for ev in result.events:
    print(f"  seq={ev.sequence:2d}  {ev.type:24s}  {dict(ev.data)}")

`loop.start` and `loop.complete` always bracket the run. `strategy.selected` records which strategy was chosen — `react` by default. `turn.start` / `turn.end` mark every iteration. `llm.call` carries token usage and latency. We'll see more events in later sections.

## 4. Tool integration

A `Tool` is a dataclass: name, description, JSON-Schema for inputs, and an async `execute(params, ctx) -> str`. The strategy passes a `ToolContext` carrying run metadata so the tool can correlate, check cancellation, or emit custom events.

In [ ]:
import json

_db = {
    "u1": {"name": "Ada", "role": "engineer"},
    "u2": {"name": "Linus", "role": "maintainer"},
}


async def lookup_user(params: dict, ctx: ToolContext) -> str:
    user_id = params["user_id"]
    record = _db.get(user_id)
    if record is None:
        return json.dumps({"error": f"unknown user_id {user_id}"})
    # Tools can use ctx for tracing — turn_number is the iteration this fired in.
    return json.dumps({**record, "_seen_in_turn": ctx.turn_number})


lookup_tool = Tool(
    name="lookup_user",
    description="Look up a user by ID. Returns name and role, or an error.",
    input_schema={
        "type": "object",
        "properties": {"user_id": {"type": "string"}},
        "required": ["user_id"],
    },
    execute=lookup_user,
)

print("Defined tool:", lookup_tool.name)

Drive a two-turn run: turn 1 the model proposes a tool call, turn 2 it produces its final answer. Each tool dispatch goes through the executor: sandbox check → registry lookup → schema validation → execute → emit `tool.start` / `tool.end`.

In [ ]:
model = MockModel(
    [
        LLMResponse(
            tool_calls=[ToolCall(id="tc1", name="lookup_user", arguments={"user_id": "u1"})],
            stop_reason="tool_use",
        ),
        LLMResponse(content="User u1 is Ada, an engineer.", stop_reason="end_turn"),
    ]
)

result = await run(
    model=model,
    tools=[lookup_tool],
    system_prompt="You are a directory assistant.",
    task="Look up user u1.",
)

print("answer:", result.content)
print("turns: ", result.turns)
print("tool_calls_made:", result.tool_calls_made)
for ev in result.events:
    if ev.type.startswith("tool."):
        print(f"  {ev.type:18s} {dict(ev.data)}")

Registries are mutable and emit their own events when tools come and go. The strategy re-reads `registry.list_schemas()` *each turn*, so a tool added mid-run becomes available on the very next call to `model.invoke`.

In [ ]:
bus = EventBus(run_id="registry-demo")
registry = ToolRegistry(tools=[greet_tool], event_bus=bus)
registry.add(lookup_tool)
registry.remove("greet")
print("registry.names() =>", registry.names())
print()
print("events from registry mutations:")
for ev in bus.events:
    print(f"  {ev.type:18s} {dict(ev.data)}")

## 5. task_complete for early exit

Plain `end_turn` works for chat-style tasks but is fragile for autonomous agents — the model might keep talking after the work is actually done. v0.4.0 added the `task_complete` builtin: a structured terminator the agent calls explicitly.

When any tool flagged `signals_completion=True` fires, the loop:

1. captures the tool's arguments as the **completion payload**
2. emits `loop.completed` carrying that payload
3. returns a `LoopResult` whose `content` is the payload's `summary`

The schema is enforced by `TaskCompleteArgs` (Pydantic, frozen).

In [ ]:
task_complete = make_task_complete_tool()
print("name:                ", task_complete.name)
print("signals_completion:  ", task_complete.signals_completion)
print("timeout_seconds:     ", task_complete.timeout_seconds)
print()
print("input schema:")
import json as _json

print(_json.dumps(task_complete.input_schema, indent=2))

`TaskCompleteArgs` is what your code reads after the loop terminates. `status` is one of `success | partial | failed`, `summary` is required, the rest are optional context for whatever runs the agent.

In [ ]:
args = TaskCompleteArgs(
    status="partial",
    summary="completed 2 of 3 steps",
    artifacts=["./step1.txt", "./step2.txt"],
    next_steps=["finish step 3"],
)
print("frozen model:")
print(args.model_dump())

# Schema enforced — invalid status raises:
from pydantic import ValidationError

try:
    TaskCompleteArgs(status="made_up", summary="x")  # type: ignore[arg-type]
except ValidationError as exc:
    print()
    print("caught ValidationError on bad status:")
    print("  ", exc.errors()[0]["msg"])

End-to-end: register `task_complete` alongside your real tools and let the model call it when it's done. The loop terminates immediately and you get the structured payload back via `state.completion_payload` (and as a `loop.completed` event).

In [ ]:
model = MockModel(
    [
        # Turn 1: do the work
        LLMResponse(
            tool_calls=[ToolCall(id="tc1", name="lookup_user", arguments={"user_id": "u2"})],
            stop_reason="tool_use",
        ),
        # Turn 2: signal completion with structured payload
        LLMResponse(
            tool_calls=[
                ToolCall(
                    id="tc2",
                    name="task_complete",
                    arguments={
                        "status": "success",
                        "summary": "Found Linus the maintainer.",
                        "artifacts": ["u2"],
                    },
                ),
            ],
            stop_reason="tool_use",
        ),
    ]
)

result = await run(
    model=model,
    tools=[lookup_tool, task_complete],
    system_prompt="Look up the user, then call task_complete when done.",
    task="Look up user u2.",
)

print("content (== summary):", result.content)
print("turns:               ", result.turns)
print("tool_calls_made:     ", result.tool_calls_made)
completed = [e for e in result.events if e.type == "loop.completed"]
print("loop.completed payload:", dict(completed[0].data))

The mechanism is generic. The strategy code in `react.py` doesn't know the name `task_complete` — it just looks for any tool whose `signals_completion` is `True`. Want a different terminator? Build your own with the same flag.

In [ ]:
async def _stop_now(params: dict, ctx: ToolContext) -> str:
    return "stopping"


stop_tool = Tool(
    name="stop_now",
    description="Custom terminator — same mechanism, different name.",
    input_schema={
        "type": "object",
        "properties": {"status": {"type": "string"}, "summary": {"type": "string"}},
        "required": ["status", "summary"],
    },
    execute=_stop_now,
    signals_completion=True,
)

model = MockModel(
    [
        LLMResponse(
            tool_calls=[
                ToolCall(
                    id="tc1",
                    name="stop_now",
                    arguments={"status": "success", "summary": "custom stop"},
                )
            ],
            stop_reason="tool_use",
        ),
    ]
)

result = await run(
    model=model,
    tools=[stop_tool],
    system_prompt="Stop when prompted.",
    task="stop",
)
print("summary content:", result.content)
print("turns:          ", result.turns)

## 6. max_turns enforcement

Without a turn cap a misbehaving model can spin forever. `max_turns` (default `25`) is a hard limit checked at the top of every turn. When breached the loop:

1. populates `state.completion_payload` with `status=failed`, `error=max_turns`
2. emits `loop.max_turns` (the legacy event)
3. emits `loop.completed` (the unified terminator event)
4. returns a `LoopResult` with `content=None` and the existing turn count

Below: a model that always proposes a tool call and never says `end_turn`. The loop stops cleanly at `max_turns=3` instead of running away.

In [ ]:
class NeverDoneModel:
    """Always returns one tool_use response — never end_turn."""

    def __init__(self) -> None:
        self.calls = 0

    async def invoke(self, messages, tools=None):
        self.calls += 1
        return LLMResponse(
            tool_calls=[
                ToolCall(id=f"tc{self.calls}", name="lookup_user", arguments={"user_id": "u1"})
            ],
            stop_reason="tool_use",
        )


result = await run(
    model=NeverDoneModel(),
    tools=[lookup_tool],
    system_prompt="You never stop.",
    task="loop forever",
    max_turns=3,
)

print("content:           ", result.content)  # None on budget breach
print("turns:             ", result.turns)
print("tool_calls_made:   ", result.tool_calls_made)

The `loop.max_turns` event records the breach details, and `loop.completed` carries the synthesized `failed` payload for downstream consumers.

In [ ]:
for ev in result.events:
    if ev.type in {"loop.max_turns", "loop.completed"}:
        print(f"  {ev.type:18s} {dict(ev.data)}")

The breach payload is constructible directly via `make_budget_breach_args` — useful in tests, or anywhere you want consistent vocabulary for cap breaches.

In [ ]:
args = make_budget_breach_args(reason="max_turns")
print(args.model_dump())

## 7. max_cost_usd enforcement

Cost caps protect the wallet. Unlike `max_turns` (a kwarg on `run()`), `max_cost_usd` is a field on `RunState` — so you set it before kicking off the loop, typically by constructing the state directly when you need fine-grained control. The check fires **before each turn**, so we never start a turn knowing we're already over budget.

When breached:

- `state.completion_payload = {status: failed, error: max_cost, ...}`
- `loop.completed` emits with `reason=max_cost` and `cost_usd=<accumulated>`
- a `LoopResult` returns with `content=None` and the existing token/cost counts

Drive it with a stub that reports `cost_usd=5.0` per call and a 3.0 cap.

In [ ]:
from arcrun._messages import system_message, user_message
from arcrun.events import EventBus as _Bus
from arcrun.registry import ToolRegistry as _Reg
from arcrun.sandbox import Sandbox as _Sandbox
from arcrun.state import RunState as _State
from arcrun.strategies.react import react_loop


class ExpensiveModel:
    """Each invoke costs $5 — busts a $3 cap on the very next turn."""

    async def invoke(self, messages, tools=None):
        return LLMResponse(
            content=None,
            tool_calls=[ToolCall(id="tc", name="lookup_user", arguments={"user_id": "u1"})],
            stop_reason="tool_use",
            cost_usd=5.0,
        )


bus = _Bus(run_id="max-cost-demo")
reg = _Reg([lookup_tool], bus)
state = _State(
    messages=[
        system_message("keep going"),
        user_message("loop until cost cap"),
    ],
    registry=reg,
    event_bus=bus,
    run_id="max-cost-demo",
    max_cost_usd=3.0,
)
sandbox = _Sandbox(SandboxConfig(), bus)

result = await react_loop(ExpensiveModel(), state, sandbox, max_turns=10)

print("content:           ", result.content)
print("turns completed:   ", result.turns)
print("accumulated cost:  $", result.cost_usd)
print("completion_payload:", state.completion_payload)

Walk the events and notice `loop.completed` carries `reason=max_cost` and the actual accumulated cost. That's enough for an external billing system to attribute the spend and abort downstream dependents.

In [ ]:
for ev in result.events:
    if ev.type == "loop.completed":
        print("loop.completed payload:", dict(ev.data))

## 8. The loop.completed event

`loop.completed` is the canonical terminator event introduced in v0.4.0. It fires in exactly three places:

| Trigger | Payload shape |
|---|---|
| A `signals_completion` tool was invoked | The full `TaskCompleteArgs` dict (status, summary, artifacts, next_steps, error) |
| `max_turns` cap breached | `{status: failed, summary: ..., error: max_turns}` |
| `max_cost_usd` cap breached | `{reason: max_cost, cost_usd: <accumulated>}` |

It does *not* fire on plain `end_turn` chat completion — that path predates SPEC-017 and stays on the older `loop.complete` event. Treat `loop.completed` as the structured terminator and `loop.complete` as the loop bookend.

Demonstrate all three triggers in one place. Capture the payload from each and compare side-by-side.

In [ ]:
def _payload_of(events: list[Event], etype: str) -> dict | None:
    for e in events:
        if e.type == etype:
            return dict(e.data)
    return None


# Trigger 1: explicit task_complete
m1 = MockModel(
    [
        LLMResponse(
            tool_calls=[
                ToolCall(
                    id="tc",
                    name="task_complete",
                    arguments={"status": "success", "summary": "finished"},
                )
            ],
            stop_reason="tool_use",
        ),
    ]
)
r1 = await run(m1, [task_complete], "sys", "task")
p1 = _payload_of(r1.events, "loop.completed")

# Trigger 2: max_turns
r2 = await run(NeverDoneModel(), [lookup_tool], "sys", "task", max_turns=2)
p2 = _payload_of(r2.events, "loop.completed")

# Trigger 3: max_cost (replay the prior path inline)
bus3 = _Bus(run_id="e8")
reg3 = _Reg([lookup_tool], bus3)
state3 = _State(
    messages=[system_message("sys"), user_message("task")],
    registry=reg3,
    event_bus=bus3,
    run_id="e8",
    max_cost_usd=2.0,
)
r3 = await react_loop(ExpensiveModel(), state3, _Sandbox(SandboxConfig(), bus3), max_turns=5)
p3 = _payload_of(r3.events, "loop.completed")

print("task_complete payload:", p1)
print("max_turns payload:    ", p2)
print("max_cost payload:     ", p3)

## 9. Event chain integrity

Every event the bus emits is hashed into a SHA-256 chain. Each `Event` carries:

- `sequence` — monotonic index starting at 0
- `prev_hash` — the previous event's `event_hash`, or `GENESIS_PREV_HASH` for sequence 0
- `event_hash` — `SHA-256(prev_hash || canonical(self))`

Mutate any field, drop any event, reorder anything — the chain breaks and `verify_chain()` tells you exactly which index. This satisfies NIST 800-53 AU-9 (Protection of Audit Information) and AU-10 (Non-Repudiation).

Notebook 07 (`arcrun/07-event-chain-verification`) goes deep on this — here we just show the surface of the API and a tamper demo.

In [ ]:
from arcrun.events import GENESIS_PREV_HASH

# Re-use the result from the very first run() up top.
model = MockModel([LLMResponse(content="done", stop_reason="end_turn")])
result = await run(model, [greet_tool], "sys", "task")

report = verify_chain(result.events)
print("valid:               ", report.valid)
print("event_count:         ", report.event_count)
print("first_broken_index:  ", report.first_broken_index)
print("genesis prev_hash:   ", result.events[0].prev_hash == GENESIS_PREV_HASH)
print("sequence linearity:  ", [e.sequence for e in result.events])
print("clean LoopResult API: result.verify_integrity().valid =", result.verify_integrity().valid)

Tamper demo. Build a forged chain by replacing one event's data — the chain detects the break and reports the offending index.

In [ ]:
from dataclasses import replace
from types import MappingProxyType

evs = list(result.events)
tampered = list(evs)
# Forge: change the data of event #1 without recomputing the hash chain.
tampered[1] = replace(tampered[1], data=MappingProxyType({"injected": "evil"}))

tamper_report = verify_chain(tampered)
print("valid:               ", tamper_report.valid)
print("first_broken_index:  ", tamper_report.first_broken_index)
print("error:               ", tamper_report.error)

`Event` is a frozen dataclass with `MappingProxyType` data, so direct mutation of an emitted event raises. The forgery above used `dataclasses.replace` to construct a *new* event — and the chain still caught it because the new event's `event_hash` was no longer derivable from the tampered fields.

In [ ]:
try:
    result.events[0].data["sneak"] = "value"  # type: ignore[index]
except TypeError as exc:
    print("frozen MappingProxyType:", exc)

try:
    result.events[0].sequence = 99  # type: ignore[misc]
except Exception as exc:
    print("frozen dataclass field:", type(exc).__name__, exc)

## 10. Strategy prompt injection

The model needs to know how to use the loop — that there's a Reason → Act → Observe cadence, that `task_complete` exists, that some tools are sandboxed. v0.4.0 added `get_strategy_prompts()` so consuming agents (typically arcagent) can pull the right guidance fragments and inject them into the system prompt without arcrun owning the prompt assembly itself.

Three things drive the output:

- `allowed_strategies` — keys for each strategy's `prompt_guidance` (defaults to `["react"]`)
- multiple strategies → adds a `strategy_selection` section
- presence of `execute_python` or `contained_execute_python` in `tool_names` → adds code-exec guidance

In [ ]:
sections = get_strategy_prompts()
print("keys:", list(sections))
print()
print("strategy_react:")
print("-" * 60)
print(sections["strategy_react"])

Pass multiple strategies and the selection guidance shows up. List code-exec tools and you get tool-specific guidance.

In [ ]:
sections = get_strategy_prompts(
    allowed_strategies=["react", "code"],
    tool_names=["execute_python", "task_complete"],
)
for key in sections:
    print(f"--- {key} ---")
    print(sections[key][:200] + ("..." if len(sections[key]) > 200 else ""))
    print()

Notice arcrun never carries guidance for tools it doesn't own. `task_complete` is covered inside the strategy's own guidance text — separate ownership boundary from agent-level orchestration tools like `spawn_task`, which live in arcagent.

## 11. run_async + RunHandle

`run()` blocks until the loop finishes. `run_async()` returns a `RunHandle` immediately and lets the loop run as a background `asyncio.Task`. The handle exposes:

| Method | Effect |
|---|---|
| `await handle.steer(message)` | Inject a user message that gets drained at the *next* turn boundary |
| `await handle.follow_up(message)` | Queue work for after the model says `end_turn` — keeps the loop running |
| `await handle.cancel()` | Set `cancel_event`, drain queues, partial result returns next |
| `await handle.result()` | Await the final `LoopResult` |
| `handle.state` | Read-only access to live `RunState` |

In [ ]:
model = MockModel(
    [
        LLMResponse(
            tool_calls=[ToolCall(id="tc1", name="lookup_user", arguments={"user_id": "u1"})],
            stop_reason="tool_use",
        ),
        LLMResponse(content="User u1 found.", stop_reason="end_turn"),
    ]
)

handle = await run_async(
    model=model,
    tools=[lookup_tool],
    system_prompt="sys",
    task="look up u1",
)

print("handle type:    ", type(handle).__name__)
print("run_id:         ", handle.state.run_id)
print("strategy_name:  ", handle.state.strategy_name or "(set by selection)")

result = await handle.result()
print("\nfinal:", result.content)
print("turns:", result.turns)

`follow_up()` queues a follow-up that fires when the model says `end_turn`. The loop drains the queue, appends the message, and continues — so a chat-style end can re-open.

In [ ]:
model = MockModel(
    [
        LLMResponse(content="first done", stop_reason="end_turn"),
        LLMResponse(content="follow-up done", stop_reason="end_turn"),
    ]
)

handle = await run_async(model, [greet_tool], "sys", "first task")
await handle.follow_up("now do the follow-up")

result = await handle.result()
print("final content:", result.content)
print("turns:        ", result.turns)

`cancel()` sets the cancel event and drains the steer/follow-up queues so partial results are clean.

In [ ]:
import asyncio as _aio

model = MockModel(
    [
        LLMResponse(
            tool_calls=[ToolCall(id="tc1", name="lookup_user", arguments={"user_id": "u1"})],
            stop_reason="tool_use",
        ),
        LLMResponse(content="never reached", stop_reason="end_turn"),
    ]
)

handle = await run_async(model, [lookup_tool], "sys", "long task")
await handle.cancel()
result = await handle.result()

print("cancel_event set:", handle.state.cancel_event.is_set())
print("final content:   ", result.content)
_ = _aio  # silence unused import

## 12. (live) one real call

Optional: drive the loop against a real Anthropic model. Skipped automatically when `ANTHROPIC_API_KEY` is missing.

In [ ]:
if not HAS_LIVE_KEY:
    print("skip — set ANTHROPIC_API_KEY in .env to run")
    raise SystemExit

from arcllm import load_model

live_model = load_model("anthropic", "claude-haiku-4-5-20251001", telemetry=True)

result = await run(
    model=live_model,
    tools=[lookup_tool, make_task_complete_tool()],
    system_prompt=(
        "You are a directory bot. Look up the user, then call task_complete with status='success'."
    ),
    task="Look up user u1 and report back.",
    max_turns=5,
)

print("answer:        ", result.content)
print("turns:         ", result.turns)
print("tokens:        ", result.tokens_used)
print("cost_usd:      ", result.cost_usd)
print("chain valid:   ", result.verify_integrity().valid)

if hasattr(live_model, "close"):
    await live_model.close()

## 13. Summary

What this notebook covered, end-to-end:

- **ReAct strategy** — Reason → Act → Observe per turn; ends on `end_turn`, `signals_completion`, or budget cap.
- **`run` / `run_async` / `RunHandle`** — blocking and non-blocking entry points with steer/follow_up/cancel semantics.
- **`Tool`, `ToolContext`, `ToolRegistry`** — dataclass tools, schema-validated inputs, mutable registry visible to the strategy on every turn.
- **`task_complete` builtin** — `make_task_complete_tool()` + `TaskCompleteArgs` give structured early exit; the loop's `signals_completion` flag is generic.
- **`max_turns`** — kwarg cap on `run()`, synthesizes `failed`/`max_turns` payload.
- **`max_cost_usd`** — `RunState` field; checked top-of-turn, fires `failed`/`max_cost`.
- **`loop.completed` event** — single canonical terminator, fires on completion tool *or* either budget breach.
- **Hash-chained events** — `Event`, `EventBus`, `verify_chain`, `GENESIS_PREV_HASH`, `ChainVerificationResult` — tamper-evident audit per AU-9/AU-10.
- **`get_strategy_prompts()`** — strategy + tool-aware guidance fragments for the consuming agent's system prompt.
- **`Strategy` ABC** — `name`, `description`, `prompt_guidance`, async `__call__`; ReAct is the default.

**Public API exercised:** `run`, `run_async`, `RunHandle`, `Strategy`, `LoopResult`, `Tool`, `ToolContext`, `ToolRegistry`, `SandboxConfig`, `Event`, `EventBus`, `verify_chain`, `GENESIS_PREV_HASH`, `ChainVerificationResult`, `get_strategy_prompts`, `make_task_complete_tool`, `TaskCompleteArgs`, `make_budget_breach_args`.

**Next:** `arcrun/02-tool-executor.ipynb` for the executor + parallel dispatch details, `arcrun/06-task-completion-budgets.ipynb` for a deep dive on completion payloads, and `arcrun/07-event-chain-verification.ipynb` for the full hash-chain treatment.